# SEC Form 4 Filings: Who is Buying and Who is Selling?

Insider transaction reporting via SEC Form 4 plays an important role in market transparency by forcing corporate officers, directors, and large shareholders to disclose their stock trades within two business days of execution. Because Form 4s contain rich, structured information—transaction dates, security details, number of shares, prices, and the insider’s relationship to the issuer—systematically extracting and aggregating these filings enables analysts and investors to detect patterns such as confidence buys, opportunistic sales, or hedging activity long before quarterly disclosures arrive. 

Automating Form 4 extraction addresses challenges of inconsistent XML/TXT layouts, varying footnote structures, and high filing volumes, ensuring that trading signals derived from insider behavior are captured accurately, updated in real time, and integrated seamlessly into quantitative models and trading dashboards. While it is possible to manually extract from the XML data, this can be a time consuming and error prone process. We will show how we can use LlamaExtract to do this automatically without having to parse the XML data.

## Dow Jones Industrial Average Companies

For this exercise, we will extract the insider transactions for all the companies in the Dow Jones Industrial Average. Let's first get the list of tickers in the Dow Jones Industrial Average using Wikipedia.

In [ ]:
%pip install pandas
%pip install lxml


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 56.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

url = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
parsed = pd.read_html(url, header=0)
dow = next(t for t in parsed if "Symbol" in t.columns)
tickers = dow["Symbol"].tolist()
tickers

['MMM',
 'AXP',
 'AMGN',
 'AMZN',
 'AAPL',
 'BA',
 'CAT',
 'CVX',
 'CSCO',
 'KO',
 'DIS',
 'GS',
 'HD',
 'HON',
 'IBM',
 'JNJ',
 'JPM',
 'MCD',
 'MRK',
 'MSFT',
 'NKE',
 'NVDA',
 'PG',
 'CRM',
 'SHW',
 'TRV',
 'UNH',
 'VZ',
 'V',
 'WMT']

Let's see an example of a Form 4 filing for MMM. Disregard how we got the URL below, we will show how to get the URL for all the Form 4 filings in the workflow below.

In [ ]:
import requests

# Use your own email below
HEADERS = {"User-Agent": "Your Name your_email@domain.com"}
example_url = "https://www.sec.gov/Archives/edgar/data/66740/000112760225005147/0001127602-25-005147.txt"
r = requests.get(example_url, headers=HEADERS)
print(r.text)

<SEC-DOCUMENT>0001127602-25-005147.txt : 20250218
<SEC-HEADER>0001127602-25-005147.hdr.sgml : 20250218
<ACCEPTANCE-DATETIME>20250218111341
ACCESSION NUMBER:		0001127602-25-005147
CONFORMED SUBMISSION TYPE:	4
PUBLIC DOCUMENT COUNT:		1
CONFORMED PERIOD OF REPORT:	20250207
FILED AS OF DATE:		20250218
DATE AS OF CHANGE:		20250218

REPORTING-OWNER:	

	OWNER DATA:	
		COMPANY CONFORMED NAME:			Chavez Rodriguez Beatriz Karina
		CENTRAL INDEX KEY:			0001890011
		ORGANIZATION NAME:           	

	FILING VALUES:
		FORM TYPE:		4
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-03285
		FILM NUMBER:		25633441

	MAIL ADDRESS:	
		STREET 1:		3M COMPANY
		STREET 2:		3M CENTER BLDG 220-09-E-02
		CITY:			ST. PAUL
		STATE:			MN
		ZIP:			55144-1000

ISSUER:		

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			3M CO
		CENTRAL INDEX KEY:			0000066740
		STANDARD INDUSTRIAL CLASSIFICATION:	SURGICAL & MEDICAL INSTRUMENTS & APPARATUS [3841]
		ORGANIZATION NAME:           	08 Industrial Applications and Services
		IRS NUMBER:

## Data Schema for Form 4 Extraction

Let's define the data schema for the Form 4 extraction. We will use Pydantic to define our schema, and use LlamaExtract to extract the data from the Form 4 filings. Note that since we are using an LLM to extract the data, we don't need to worry about the exact layout/schema for the XML files. 

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional


class Address(BaseModel):
    street1: str = Field(description="Business or owner street address line 1")
    street2: Optional[str] = Field(None, description="Street address line 2")
    city: str
    state: str
    zipCode: str


class Issuer(BaseModel):
    ticker: str = Field(description="Ticker of the issuer")
    name: str = Field(description="Name of the issuer")
    cik: str = Field(description="10-digit SEC CIK of the issuer")
    sic: Optional[str] = Field(
        None, description="Standard Industrial Classification code"
    )
    stateOfIncorporation: Optional[str]


class ReportingOwner(BaseModel):
    name: str = Field(description="Name of the reporting owner")
    cik: Optional[str] = Field(
        None, description="CIK of the reporting owner (if available)"
    )
    address: Address
    relationship: Optional[str] = Field(
        None, description="One of: Officer, Director, 10% owner, Affiliate, Other"
    )


class UnderlyingSecurity(BaseModel):
    title: str = Field(description="Title of the underlying security")
    conversionOrExercisePrice: Optional[float] = Field(None)
    conversionOrExerciseDate: Optional[str] = Field(
        None, description="Exercise date in ISO 8601 format (YYYY-MM-DD)"
    )
    underlyingSecurityShares: Optional[int] = Field(
        None, description="Number of shares underlying each derivative"
    )


class NonDerivativeTransaction(BaseModel):
    securityTitle: str = Field(description="Title of the security")
    transactionDate: str = Field(
        description="Transaction date in ISO 8601 format (YYYY-MM-DD)"
    )
    transactionCode: str = Field(
        description="Code indicating transaction type (A, D, P, S, etc.)"
    )
    transactionShares: float
    transactionPricePerShare: float
    totalTransactionValue: Optional[float] = Field(
        None, description="transactionShares × transactionPricePerShare"
    )
    ownershipNature: str = Field(description="e.g. Direct, Indirect")
    footnoteIds: Optional[List[str]] = Field(
        None, description="References into the footnotes block"
    )


class DerivativeTransaction(BaseModel):
    securityTitle: str
    transactionDate: str = Field(
        description="Transaction date in ISO 8601 format (YYYY-MM-DD)"
    )
    transactionCode: str
    numberOfDerivativeSecurities: int
    exercisePrice: Optional[float]
    expirationDate: Optional[str] = Field(
        description="Expiration date in ISO 8601 format (YYYY-MM-DD)"
    )
    underlyingSecurity: UnderlyingSecurity
    directOrIndirect: str = Field(description="Direct or Indirect")
    footnoteIds: Optional[List[str]]


class Footnote(BaseModel):
    id: str = Field(description="Footnote identifier")
    text: str = Field(description="Full footnote text")


class Form4Metadata(BaseModel):
    cik: str = Field(description="CIK of the filer")
    accessionNumber: str
    primaryDocument: str = Field(description="Filename of the primary XML/TXT")
    filingDate: str = Field(description="Filing date in ISO 8601 format (YYYY-MM-DD)")
    reportDate: str = Field(
        description="Period of report date in ISO 8601 format (YYYY-MM-DD)"
    )


class Form4(BaseModel):
    metadata: Form4Metadata
    issuer: Issuer
    reportingOwners: List[ReportingOwner]
    nonDerivativeTransactions: List[NonDerivativeTransaction]
    derivativeTransactions: List[DerivativeTransaction]
    footnotes: Optional[List[Footnote]] = Field(default=None)

### Creating the LlamaExtract Agent

In [ ]:
from dotenv import load_dotenv
from llama_cloud_services import LlamaExtract


# Load environment variables (put LLAMA_CLOUD_API_KEY in your .env file)
load_dotenv(override=True)

# Optionally, add your project id/organization id
extract = LlamaExtract(show_progress=False)

In [ ]:
from llama_cloud.core.api_error import ApiError

try:
    existing_agent = extract.get_agent(name="Form4Extractor")
    if existing_agent:
        # Deletion can take some time since all underlying files will be purged
        extract.delete_agent(existing_agent.id)
except ApiError as e:
    if e.status_code == 404:
        pass
    else:
        raise

agent = extract.create_agent("Form4Extractor", data_schema=Form4)

### Example Extraction

In [ ]:
from llama_cloud_services.extract import SourceText

agent.extract(SourceText(text_content=r.text)).data

{'metadata': {'cik': '0001890011',
  'accessionNumber': '0001127602-25-005147',
  'primaryDocument': 'form4.xml',
  'filingDate': '2025-02-18',
  'reportDate': '2025-02-07'},
 'issuer': {'ticker': 'MMM',
  'name': '3M CO',
  'cik': '0000066740',
  'sic': '3841',
  'stateOfIncorporation': 'DE'},
 'reportingOwners': [{'name': 'Chavez Rodriguez Beatriz Karina',
   'cik': '0001890011',
   'address': {'street1': '3M CENTER',
    'street2': None,
    'city': 'ST. PAUL',
    'state': 'MN',
    'zipCode': '55144'},
   'relationship': 'Officer'}],
 'nonDerivativeTransactions': [{'securityTitle': 'Common Stock',
   'transactionDate': '2025-02-07',
   'transactionCode': 'M',
   'transactionShares': 3569.0,
   'transactionPricePerShare': 149.87,
   'totalTransactionValue': 535186.03,
   'ownershipNature': 'Direct',
   'footnoteIds': None},
  {'securityTitle': 'Common Stock',
   'transactionDate': '2025-02-07',
   'transactionCode': 'F',
   'transactionShares': 1108.0,
   'transactionPricePerShare'

## Workflow for Extracting Insider Transactions from SEC Form 4

Now that we have defined the LlamaExtract agent, we can create a workflow to scalably extract the insider transactions from SEC Form 4 filings for all the companies in the Dow Jones Industrial Average. We will use all the filings during 2024 and the first quarter of 2025.

⚠ Note that we need to limit ourselves to within 10reqs/second when hitting the SEC data so as not to get IP banned. We will use AsyncLimited for aiolimiter in our workflow for this.

In [ ]:
%pip install aiolimiter


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from aiolimiter import AsyncLimiter
import httpx

from llama_index.core.workflow import (
    Context,
    Workflow,
    Event,
    StartEvent,
    StopEvent,
    step,
)


class SECForm4URL(Event):
    "Form 4 URL to extract from"
    ticker: str
    cik: str
    file_date: str
    accession_no: str
    url: str


class ExtractedData(Event):
    "Data extracted from SEC Form 4"
    extracted: Form4


class Form4Extraction(Workflow):
    def __init__(self, start_date: str, end_date: str, verbose=False, timeout=480):
        super().__init__(verbose=verbose, timeout=timeout)
        self.start_date = start_date
        self.end_date = end_date
        self.limiter = AsyncLimiter(max_rate=9, time_period=1)
        self.result = None
        self._http_client = httpx.AsyncClient(headers=HEADERS, timeout=60)

    async def __aenter__(self):
        await self._http_client.__aenter__()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self._http_client.__aexit__(exc_type, exc_val, exc_tb)

    @step
    async def get_djia_companies(self, ctx: Context, ev: StartEvent) -> SECForm4URL:
        """
        Fetch all Form 4 URLs for companies in the DJIA index
        between (start_date, end_date).
        """
        url = "https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average"
        parsed = pd.read_html(url, header=0)
        dow = next(t for t in parsed if "Symbol" in t.columns)
        tickers = dow["Symbol"].tolist()
        if self._verbose:
            print(f"Number of companies to extract: {len(tickers)}")
        mapping_resp = await self._http_client.get(
            "https://www.sec.gov/files/company_tickers.json"
        )
        mapping_resp.raise_for_status()
        mapping = mapping_resp.json()
        ticker_to_cik = {
            v["ticker"]: str(v["cik_str"]).zfill(10) for v in mapping.values()
        }
        for ticker in tickers:
            cik = ticker_to_cik[ticker]
            url = f"https://data.sec.gov/submissions/CIK{cik}.json"
            async with self.limiter:
                resp = await self._http_client.get(url)
                resp.raise_for_status()

            recent = resp.json().get("filings", {}).get("recent", {})
            forms = recent.get("form", [])
            dates = recent.get("filingDate", [])
            accession = recent.get("accessionNumber", [])
            urls = []
            for f, d, acc in zip(forms, dates, accession):
                if f == "4" and self.start_date <= d <= self.end_date:
                    idx = acc.replace("-", "")
                    urls.append(
                        SECForm4URL(
                            ticker=ticker,
                            cik=cik,
                            file_date=d,
                            accession_no=acc,
                            url=f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{idx}/{acc}.txt",
                        )
                    )
        if self._verbose:
            print(f"Number of forms to extract: {len(urls)}")
        await ctx.set("num_forms", len(urls))
        for url in urls:
            ctx.send_event(url)

    @step(num_workers=5)
    async def fetch_sec_form4_filings(self, ev: SECForm4URL) -> ExtractedData:
        """
        Download form 4 from the SEC website, and use LlamaExtract agent defined earlier
        to extract the data from the form.
        """
        async with self.limiter:
            r = await self._http_client.get(ev.url)
            r.raise_for_status()
        extracted = await agent.aextract(
            SourceText(text_content=f"ticker: {ev.ticker}\n\nform 4: {r.text}")
        )
        return ExtractedData(extracted=extracted.data)

    @step
    async def collect_results(
        self, ctx: Context, ev: ExtractedData
    ) -> StopEvent | None:
        """
        Collect all the data from the SEC forms and put into a single Pandas DataFrame.
        """
        num_forms = await ctx.get("num_forms")
        results = ctx.collect_events(ev, [ExtractedData] * num_forms)
        if results is None:
            return
        self.result = [r.extracted.model_dump(mode="json") for r in results]
        if self._verbose:
            print(f"{len(self.result)} results dumped into Pandas dataframe.")
        return StopEvent()

In [ ]:
extraction = Form4Extraction(
    start_date="2024-01-01", end_date="2025-03-31", verbose=False
)
await extraction.run()

Uploading files:   0%|          | 0/1 [10:28<?, ?it/s]


## Create a Pandas DataFame for analysis

Now that we have the data extracted, we can create a Pandas DataFrame with the relevant subset of fields that we can use for analysis.

In [ ]:
# 1) map SEC transaction codes to human labels
# Add any other codes you need to consider in your analysis...
CODE_MAP = {
    "S": "Sale",
    "P": "Proposed Sale",
    "A": "Acquisition",
    "D": "Disposition",
}


def forms_to_dataframe(forms_json: list[dict]) -> pd.DataFrame:
    records = []
    for form in forms_json:
        filing_date = form["metadata"]["filingDate"]
        # loop each reporting owner (usually one)
        for owner in form.get("reportingOwners", []):
            # non-derivative transactions
            for tx in form.get("nonDerivativeTransactions", []):
                records.append(
                    {
                        "Ticker": form["issuer"]["ticker"],
                        "Company Name": form["issuer"]["name"],
                        "Insider Name": owner["name"],
                        "Relationship": owner.get("relationship"),
                        "Transaction Date": pd.to_datetime(tx["transactionDate"]),
                        "Transaction": CODE_MAP.get(
                            tx["transactionCode"], tx["transactionCode"]
                        ),
                        "Cost": tx["transactionPricePerShare"],
                        "#Shares": tx["transactionShares"],
                        "Value ($)": tx.get(
                            "totalTransactionValue",
                            tx["transactionShares"] * tx["transactionPricePerShare"],
                        ),
                        "#Shares Total": tx.get("postTransactionShares"),
                        "Filing Date": pd.to_datetime(filing_date),
                    }
                )

            # derivative transactions
            for tx in form.get("derivativeTransactions", []):
                records.append(
                    {
                        "Ticker": form["issuer"]["ticker"],
                        "Company Name": form["issuer"]["name"],
                        "Insider Name": owner["name"],
                        "Relationship": owner.get("relationship"),
                        "Transaction Date": pd.to_datetime(tx["transactionDate"]),
                        "Transaction": CODE_MAP.get(
                            tx["transactionCode"], tx["transactionCode"]
                        ),
                        "Cost": tx.get(
                            "transactionPricePerShare", tx.get("exercisePrice")
                        ),
                        "#Shares": tx.get("numberOfDerivativeSecurities"),
                        "Value ($)": None,
                        "#Shares Total": None,
                        "Filing Date": pd.to_datetime(filing_date),
                    }
                )

    return pd.DataFrame(records)


df = forms_to_dataframe(extraction.result)
print(f"Number of records: {len(df)}")
df.head()

Number of records: 430


,Ticker,Company Name,Insider Name,Relationship,Transaction Date,Transaction,Cost,#Shares,Value ($),#Shares Total,Filing Date
0,WMT,Walmart Inc.,Walton Family Holdings Trust,10% owner,2025-03-26,J,0.0,297000.0,0.0,None,2025-03-28
1,WMT,Walmart Inc.,Walton Steuart L,Director,2025-03-26,J,0.0,27000.0,0.0,None,2025-03-28
2,WMT,Walmart Inc.,Walton Steuart L,Director,2025-03-28,G,0.0,27000.0,0.0,None,2025-03-28
3,WMT,Walmart Inc.,Bartlett Daniel J,Officer,2025-03-24,G,0.0,2304.0,0.0,None,2025-03-24
4,WMT,Walmart Inc.,Penner Gregory Boyd,Director,2025-03-26,J,0.0,27000.0,0.0,None,2025-03-28
